# Evaluation - An explanation

This notebook explains the the evaluation process with a few examples.

### Preliminary ground truth

For the purpose of testing, I used a small preliminary ground-truth file:

```text
lion;25F23(LION);43C111232;48A9843;
bridge;46C112;46C1122;;
angel;11GG193;11GG1923;11G193;11G32
```

The first column has the query text. The remaining columns are the relevant Iconclass notations for that query.

In [41]:
sample_ground_truth = {
    "lion": ["25F23(LION)", "43C111232", "48A9843"],
    "bridge": ["46C112", "46C1122"],
    "angel": ["11GG193", "11GG1923", "11G193", "11G32"]
}

sample_ground_truth

{'lion': ['25F23(LION)', '43C111232', '48A9843'],
 'bridge': ['46C112', '46C1122'],
 'angel': ['11GG193', '11GG1923', '11G193', '11G32']}

## Evaluation files

The model scripts save their results in `model_results.jsonl`. Each line contains the result of one model for one query. The evaluation script reads this file, calculates the metrics, and writes the output to `evaluation_results.jsonl`.

The hierarchy-based part also needs `iconclass_hierarchy.db`. This database stores the Iconclass codes, their labels, their parent codes, and their depth in the hierarchy. The parent and depth values are needed for the Wu-Palmer calculation.

### `load_jsonl` and `write_jsonl`

`load_jsonl` reads a JSONL file line by line and returns a list of records.

`write_jsonl` writes a list of records to a JSONL file. Each record is written as one JSON object on one line.

In [42]:
import json

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records


def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

## Metric functions

This section gives a short explanation of the metric functions used in the evaluation code. These functions calculate the numerical values saved in `evaluation_results.jsonl`.

### `precision_recall_f1`

This function calculates precision, recall, and F1-score.

The formulas are:
$$
\text{Precision}
=
\frac{
\text{number of correct predicted codes}
}{
\text{number of predicted codes}
}
$$

$$
\text{Recall}
=
\frac{
\text{number of correct predicted codes}
}{
\text{number of ground-truth codes}
}
$$

$$
F_1
=
\frac{
2 \times \text{Precision} \times \text{Recall}
}{
\text{Precision} + \text{Recall}
}
$$

In [43]:
def precision_recall_f1(predicted_codes, ground_truth_codes):
    predicted_set = set(predicted_codes)
    ground_truth_set = set(ground_truth_codes)

    if not predicted_set:
        precision = 0.0
    else:
        precision = len(predicted_set & ground_truth_set) / len(predicted_set)

    if not ground_truth_set:
        recall = 0.0
    else:
        recall = len(predicted_set & ground_truth_set) / len(ground_truth_set)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

### `r_precision`

This function calculates R-precision. Here, `R` is the number of ground-truth codes for the query.

The formula is:

$$
\text{R-precision}
=
\frac{
\text{number of correct codes in the top } R \text{ predictions}
}{
R
}
$$

In [44]:
def r_precision(predicted_codes, ground_truth_codes):
    r = len(ground_truth_codes)

    if r == 0:
        return 0.0

    top_r = predicted_codes[:r]
    return len(set(top_r) & set(ground_truth_codes)) / r

### `average_precision`

This function calculates average precision for one query. It rewards correct predictions more when they appear earlier in the ranked list.

The formula is:

$$
AP
=
\frac{
\sum_{k \in H} \text{Precision@}k
}{
|G|
}
$$

In [45]:
def average_precision(predicted_codes, ground_truth_codes):
    ground_truth_set = set(ground_truth_codes)

    if not ground_truth_set:
        return 0.0

    hits = 0
    precision_sum = 0.0
    already_found = set()

    for rank, code in enumerate(predicted_codes, start=1):
        if code in ground_truth_set and code not in already_found:
            hits += 1
            already_found.add(code)
            precision_sum += hits / rank

    return precision_sum / len(ground_truth_set)

### How the hierarchy database is built

The hierarchy database is built before the evaluation step. The script reads the Iconclass data and creates a SQLite database called `iconclass_hierarchy.db`.

In the final setup:

-> normal Iconclass notations are kept <br>
-> notations with additions such as `(+...)` are only kept if they are listed in `used_notation_keys.txt`

With this rule the candidate set is kept consistent across the models.

The database table used by the evaluation has this structure:

```sql
CREATE TABLE iconclass (
    notation TEXT NOT NULL,
    lang TEXT NOT NULL,
    label TEXT NOT NULL,
    parent TEXT,
    depth INTEGER NOT NULL,
    PRIMARY KEY (notation, lang)
);
```

For Wu-Palmer, the most important part is that each code has a stored parent and depth. From the parent, we can move upward in the hierarchy until we reach the root.

### How ancestors are reached

-> For Wu-Palmer similarity , we need to find the deepest shared ancestor of two Iconclass codes.

The exact ancestors depend on the Iconclass hierarchy, but the logic is always the same:

1. Start with the code itself
2. Look up its parent in the database
3. Add the parent to the ancestor list
4. Move to the parent
5. Repeat until there is no parent anymore

The evaluation stores the ancestors in a dictionary where:

-> the key is the ancestor code
-> the value is the depth of that ancestor

In [46]:
import sqlite3

class IconclassHierarchy:
    def __init__(self, db_path):
        self.con = sqlite3.connect(db_path)
        self.cache = {}

    def get_parent_depth(self, code):
        if code in self.cache:
            return self.cache[code]

        cur = self.con.cursor()
        cur.execute(
            """
            SELECT parent, depth
            FROM iconclass
            WHERE notation=?
            LIMIT 1
            """,
            (code,)
        )

        row = cur.fetchone()

        if row is None:
            self.cache[code] = None
            return None

        parent, depth = row
        self.cache[code] = (parent, depth)
        return parent, depth

    def ancestors(self, code):
        result = {}
        current = code

        while current:
            parent_depth = self.get_parent_depth(current)

            if parent_depth is None:
                break

            parent, depth = parent_depth
            result[current] = depth
            current = parent

        return result

    def close(self):
        self.con.close()

## Wu-Palmer similarity

Wu-Palmer similarity compares two individual concepts in a hierarchy.

It looks at:

-> the depth of the first code
-> the depth of the second code
-> the depth of their closest shared ancestor

The closest shared ancestor (LCS = "Leas Common Subsumer") is the most specific parent concept that both codes have in common.

The formula is:

$$
\text{Wu-Palmer Similarity}(code_1, code_2) =
\frac{2 \times depth(LCS(code_1, code_2))}
{depth(code_1) + depth(code_2)}
$$

The score is between 0 and 1.

-> 1.0 means the codes are the same <br>
-> A high score means the codes are close in the hierarchy <br>
-> A low score means the codes are far apart <br>
-> 0.0 means that no useful shared ancestor was found in the database

(In the implementation, the depths are shifted by `+1`, because the root depth in the database starts at `0`. Adding `1` avoids problems with concepts closer to the root.)

In [47]:
class IconclassHierarchyWithWuPalmer(IconclassHierarchy):
    def wu_palmer(self, code1, code2):
        if code1 == code2:
            return 1.0

        info1 = self.get_parent_depth(code1)
        info2 = self.get_parent_depth(code2)

        if info1 is None or info2 is None:
            return 0.0

        _, depth1 = info1
        _, depth2 = info2

        ancestors1 = self.ancestors(code1)
        ancestors2 = self.ancestors(code2)

        common = set(ancestors1.keys()) & set(ancestors2.keys())

        if not common:
            return 0.0

        lca = max(common, key=lambda c: ancestors1[c])
        lca_depth = ancestors1[lca]

        depth1 = depth1 + 1
        depth2 = depth2 + 1
        lca_depth = lca_depth + 1

        return (2 * lca_depth) / (depth1 + depth2)

### Example: Pairwise Wu-Palmer scores

The next cell compares some of the test ground-truth codes pair by pair. 

(-> It only works if the hierarchy database is available in the same folder.)

For example, it can show whether `46C112` and `46C1122` are close in the hierarchy.

IMPORTANT NOTE: After this point you need the Iconlcass hierarchical database in the same folder named as "iconclass_hierarchy.db", or else you'll get an error.

In [ ]:
import os
BASE_DIR = os.getcwd()
HIERARCHY_DB_PATH = os.path.join(BASE_DIR, "iconclass_hierarchy.db")

In [49]:
if os.path.exists(HIERARCHY_DB_PATH):
    hierarchy = IconclassHierarchyWithWuPalmer(HIERARCHY_DB_PATH)

    pairs_to_check = [
        ("46C112", "46C1122"),
        ("25F23(LION)", "43C111232"),
        ("11GG193", "11GG1923"),
        ("11GG193", "11G32")
    ]

    for code1, code2 in pairs_to_check:
        score = hierarchy.wu_palmer(code1, code2)
        print(f"{code1}  <->  {code2}: {score:.4f}")

    hierarchy.close()
else:
    print("iconclass_hierarchy.db was not found in this folder.")

46C112  <->  46C1122: 0.9231
25F23(LION)  <->  43C111232: 0.0000
11GG193  <->  11GG1923: 0.8000
11GG193  <->  11G32: 0.5000


## Why pairwise Wu-Palmer is not enough

The original Wu-Palmer similarity compares two concepts:

```text
one code  <->  one code
```

But the model evaluation has lists:

```text
ground truth codes  <->  predicted codes
```

For example, for `bridge`:

```text
ground truth = [46C112, 46C1122]
predictions  = [46C112, 46C11211, 43C7116, ...]
```

So we need a way to summarize several pairwise Wu-Palmer scores into one score for the whole query.

For this reason, the evaluation file calculates Wu-Palmer in two directions:

1. ground truth to prediction
2. prediction to ground truth

Then it averages these two directions, to get a bigger idea of the picture.

## Ground-truth to prediction: `wu_palmer_gt_to_pred`

This direction starts from the ground truth.

For each ground-truth code, it asks:

-> Is there at least one predicted code that is close to this correct code?

Then it takes the best Wu-Palmer score for each ground-truth code (each query) and averages these scores.

Formula:

```text
wu_palmer_gt_to_pred =
average over every ground-truth code of:
    best Wu-Palmer score between that ground-truth code and any predicted code
```

In mathematical notation:

$$
\text{Score}(G, P) =
\frac{1}{|G|} \sum_{g \in G} \max_{p \in P} \operatorname{WUP}(g, p)
$$

where:

- `G` is the list of ground-truth codes (queries)
- `P` is the list of predicted codes
- `WUP(g, p)` is the pairwise Wu-Palmer score between one ground-truth code and one predicted code

This score aims to tells us whether the important correct concepts were found, at least approximately, in the hierarchy.

In [50]:
def wu_palmer_gt_to_pred(predicted_codes, ground_truth_codes, hierarchy):
    if not predicted_codes or not ground_truth_codes:
        return 0.0

    scores = []

    for gt_code in ground_truth_codes:
        best_score = max(
            hierarchy.wu_palmer(gt_code, pred_code)
            for pred_code in predicted_codes
        )
        scores.append(best_score)

    return sum(scores) / len(scores)

### Simple example for `gt_to_pred`

For `bridge`, the ground truth contains two codes:

```text
ground truth = [46C112, 46C1122]
```

Suppose a model predicts:

```text
predictions = [46C112, 46C11211, 43C7116]
```

The `gt_to_pred` score checks each ground-truth code separately:

```text
46C112  -> find the closest prediction
46C1122 -> find the closest prediction
```

If each ground-truth code has a close prediction, this score becomes high.

This means:

-> The model covered the relevant bridge concepts well in the hierarchy.


In [51]:
if os.path.exists(HIERARCHY_DB_PATH):
    hierarchy = IconclassHierarchyWithWuPalmer(HIERARCHY_DB_PATH)

    bridge_ground_truth = ["46C112", "46C1122"]
    bridge_predictions = ["46C112", "46C11211", "43C7116"]

    score = wu_palmer_gt_to_pred(
        bridge_predictions,
        bridge_ground_truth,
        hierarchy
    )

    print(f"Bridge gt_to_pred score: {score:.4f}")

    hierarchy.close()
else:
    print("iconclass_hierarchy.db was not found in this folder.")

Bridge gt_to_pred score: 0.9615


## Prediction to ground truth: `wu_palmer_pred_to_gt`

This direction starts from the predictions.

For each predicted code, it asks:

-> Is this prediction close to at least one correct ground-truth code?

Then it takes the best Wu-Palmer score for each prediction and averages these best scores.

Formula:

```text
wu_palmer_pred_to_gt =
average over every predicted code of:
    best Wu-Palmer score between that predicted code and any ground-truth code
```

In mathematical notation:

$$
\frac{1}{|P|}
\sum_{p \in P}
\max_{g \in G}
\operatorname{WUP}(p, g)
$$

This score tells us whether the model's returned predictions are actually close to the correct concepts.

This is important because `gt_to_pred` alone can look good even if the model also returns many unrelated predictions.

For example:

```text
ground truth = [46C112]
predictions  = [46C112, unrelated code, unrelated code, unrelated code]
```

The ground-truth code found a perfect match, so `gt_to_pred` can still be high. But the prediction list also contains many bad predictions. `pred_to_gt` catches this by checking every predicted code.

In [52]:
def wu_palmer_pred_to_gt(predicted_codes, ground_truth_codes, hierarchy):
    if not predicted_codes or not ground_truth_codes:
        return 0.0

    scores = []

    for pred_code in predicted_codes:
        best_score = max(
            hierarchy.wu_palmer(pred_code, gt_code)
            for gt_code in ground_truth_codes
        )
        scores.append(best_score)

    return sum(scores) / len(scores)

### Simple example for `pred_to_gt`

Using the same `bridge` example:

```text
ground truth = [46C112, 46C1122]
predictions  = [46C112, 46C11211, 43C7116]
```

The `pred_to_gt` score checks each prediction separately:

```text
46C112   -> find the closest ground-truth code
46C11211 -> find the closest ground-truth code
43C7116  -> find the closest ground-truth code
```

If all predictions are close to the ground truth, this score becomes high.

This means:

-> The model's prediction list is clean and relevant in the hierarchy.


In [53]:
if os.path.exists(HIERARCHY_DB_PATH):
    hierarchy = IconclassHierarchyWithWuPalmer(HIERARCHY_DB_PATH)

    bridge_ground_truth = ["46C112", "46C1122"]
    bridge_predictions = ["46C112", "46C11211", "43C7116"]

    score = wu_palmer_pred_to_gt(
        bridge_predictions,
        bridge_ground_truth,
        hierarchy
    )

    print(f"Bridge pred_to_gt score: {score:.4f}")

    hierarchy.close()
else:
    print("iconclass_hierarchy.db was not found in this folder.")

Bridge pred_to_gt score: 0.6703


## Why both directions?

=>The two directions answer different questions.

### `gt_to_pred`

This asks:

-> Did the predictions cover the ground-truth concepts?

This directly supports the main goal; to find the relevant Iconclass codes.

### `pred_to_gt`

This asks:

-> Are the predicted codes close to the ground truth?

A model may cover one correct concept but still return many unrelated predictions. That'S why we look at this.

### Mean of both directions

The final `wu_palmer_mean` is the average of the two:

$$
\text{wu\_palmer\_mean}
=
\frac{
\text{wu\_palmer\_gt\_to\_pred}
+
\text{wu\_palmer\_pred\_to\_gt}
}{2}
$$

This gives a broader hierarchy-aware score.

In simple words:

gt_to_pred = Did we find the important correct things? <br>
pred_to_gt = Are the things we found actually relevant? <br>
mean       = Balanced overall hierarchy-aware score


In this case, `gt_to_pred` is especially important because it shows how well the ground-truth meanings are covered. However, I think the mean is also useful because it gives us a way to include a score for the overall quality of the predictions.

In [54]:
def average_best_wu_palmer(predicted_codes, ground_truth_codes, hierarchy):
    if not predicted_codes or not ground_truth_codes:
        return {
            "wu_palmer_gt_to_pred": 0.0,
            "wu_palmer_pred_to_gt": 0.0,
            "wu_palmer_mean": 0.0
        }

    gt_to_pred_scores = []

    for gt_code in ground_truth_codes:
        best_score = max(
            hierarchy.wu_palmer(gt_code, pred_code)
            for pred_code in predicted_codes
        )
        gt_to_pred_scores.append(best_score)

    pred_to_gt_scores = []

    for pred_code in predicted_codes:
        best_score = max(
            hierarchy.wu_palmer(pred_code, gt_code)
            for gt_code in ground_truth_codes
        )
        pred_to_gt_scores.append(best_score)

    gt_to_pred_avg = sum(gt_to_pred_scores) / len(gt_to_pred_scores)
    pred_to_gt_avg = sum(pred_to_gt_scores) / len(pred_to_gt_scores)

    return {
        "wu_palmer_gt_to_pred": gt_to_pred_avg,
        "wu_palmer_pred_to_gt": pred_to_gt_avg,
        "wu_palmer_mean": (gt_to_pred_avg + pred_to_gt_avg) / 2
    }

## Example with the three test queries

The next cell shows how the three Wu-Palmer values can be calculated for `lion`, `bridge`, and `angel`.

The prediction lists below are only example lists to demonstrate the evaluation logic. In the real evaluation, these predictions come from `model_results.jsonl`.

In [55]:
sample_predictions = {
    "lion": ["25F23(LION)", "25F23(LION)(+0)", "43C1112321"],
    "bridge": ["46C112", "46C11211", "43C7116"],
    "angel": ["11GG193", "11GG192", "11G32"]
}

if os.path.exists(HIERARCHY_DB_PATH):
    hierarchy = IconclassHierarchyWithWuPalmer(HIERARCHY_DB_PATH)

    for query in sample_ground_truth:
        gt_codes = sample_ground_truth[query]
        pred_codes = sample_predictions[query]

        scores = average_best_wu_palmer(
            pred_codes,
            gt_codes,
            hierarchy
        )

        print(f" \nQuery: {query}")
        print(f"Ground truth: {gt_codes}")
        print(f"Predictions : {pred_codes}")
        print(f"GT to prediction : {scores['wu_palmer_gt_to_pred']:.4f}")
        print(f"Prediction to GT : {scores['wu_palmer_pred_to_gt']:.4f}")
        print(f"Wu-Palmer mean  : {scores['wu_palmer_mean']:.4f}")

    hierarchy.close()
else:
    print("iconclass_hierarchy.db was not found in this folder.")

 
Query: lion
Ground truth: ['25F23(LION)', '43C111232', '48A9843']
Predictions : ['25F23(LION)', '25F23(LION)(+0)', '43C1112321']
GT to prediction : 0.6883
Prediction to GT : 0.6491
Wu-Palmer mean  : 0.6687
 
Query: bridge
Ground truth: ['46C112', '46C1122']
Predictions : ['46C112', '46C11211', '43C7116']
GT to prediction : 0.9615
Prediction to GT : 0.6703
Wu-Palmer mean  : 0.8159
 
Query: angel
Ground truth: ['11GG193', '11GG1923', '11G193', '11G32']
Predictions : ['11GG193', '11GG192', '11G32']
GT to prediction : 0.9256
Prediction to GT : 0.9778
Wu-Palmer mean  : 0.9517


## Interpretation of the Wu-palmer similarity scores

### `wu_palmer_gt_to_pred`

How well the ground-truth codes are covered by the predictions.

-> High value: For most ground-truth codes, the model returned the exact code or a hierarchically close code.

-> Low value: Some ground-truth concepts were not covered well, even approximately.

### `wu_palmer_pred_to_gt`

How close the predicted codes are to the ground truth.

-> High value: Most predictions are close to at least one relevant ground-truth code.

-> Low value: The model returned predictions that are far from the relevant concepts.

### `wu_palmer_mean`

Average of the two directions.

-> High value: The model both covered the ground truth and kept its predictions close to the relevant concepts.

-> Low value: Either the ground truth was not covered well, or many predictions were not close to the ground truth, or both.


### `evaluate_record`

This function evaluates one model result record. It takes the predicted codes and the ground-truth codes from one query, calculates all metrics, and returns one evaluation record.

In [56]:
def evaluate_record(record, hierarchy):
    predicted_codes = record.get("predicted_codes", [])
    ground_truth_codes = record.get("ground_truth_codes", [])

    precision, recall, f1 = precision_recall_f1(
        predicted_codes,
        ground_truth_codes
    )

    wu_palmer_scores = average_best_wu_palmer(
        predicted_codes,
        ground_truth_codes,
        hierarchy
    )

    evaluation = {
        "source_run_id": record.get("run_id"),
        "model": record.get("model"),
        "model_name": record.get("model_name"),
        "query": record.get("query"),
        "top_n": record.get("top_n"),
        "ground_truth_codes": ground_truth_codes,
        "predicted_codes": predicted_codes,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "r_precision": r_precision(predicted_codes, ground_truth_codes),
        "map": average_precision(predicted_codes, ground_truth_codes),
        "wu_palmer_gt_to_pred": wu_palmer_scores["wu_palmer_gt_to_pred"],
        "wu_palmer_pred_to_gt": wu_palmer_scores["wu_palmer_pred_to_gt"],
        "wu_palmer_mean": wu_palmer_scores["wu_palmer_mean"],
        "ground_truth_codes_missing_from_corpus": record.get(
            "ground_truth_codes_missing_from_corpus",
            []
        )
    }

    return evaluation

## Example - evaluation record

For every model and every query, the evaluation script creates one record in `evaluation_results.jsonl`.

```json
{
  "model": "bm25",
  "query": "bridge",
  "ground_truth_codes": ["46C112", "46C1122"],
  "predicted_codes": ["46C112", "46C11211", "43C7116"],
  "precision": 0.3333,
  "recall": 0.5000,
  "f1": 0.4000,
  "r_precision": 0.5000,
  "map": 0.5000,
  "wu_palmer_gt_to_pred": 0.9000,
  "wu_palmer_pred_to_gt": 0.7500,
  "wu_palmer_mean": 0.8250
}
```

(The numbers above are only an example. The real values depend on the actual predictions and the actual Iconclass hierarchy.)

### `summarize_by_model`

This function groups the evaluation records by model and calculates the average score of each metric for each model, so the overall model performance can be summarized and visualized with graphs.

The formula for each model and metric is:

$$
\text{Average metric score}
=
\frac{
\sum_{i=1}^{N} \text{metric score}_i
}{
N
}
$$

In [57]:
from collections import defaultdict

def summarize_by_model(records):
    metrics = [
        "precision",
        "recall",
        "f1",
        "r_precision",
        "map",
        "wu_palmer_mean",
        "wu_palmer_gt_to_pred",
        "wu_palmer_pred_to_gt"
    ]

    scores_by_model = defaultdict(lambda: defaultdict(list))

    for record in records:
        model = record.get("model")

        if not model:
            continue

        for metric in metrics:
            value = record.get(metric)

            if value is not None:
                scores_by_model[model][metric].append(value)

    models = sorted(scores_by_model.keys())
    summary = {}

    for model in models:
        summary[model] = {}

        for metric in metrics:
            values = scores_by_model[model][metric]

            if values:
                summary[model][metric] = sum(values) / len(values)
            else:
                summary[model][metric] = 0.0

    return summary, metrics, models

### Graph functions

`plot_evaluation_results` creates the average metric graphs for each model.

`plot_query_performance` creates one graph per metric and shows the score of each query. This makes it possible to see which queries are easier or harder for each model.